# Unidad 26 — Tabla hash

Semana 3 · Clase 2 · Fuente: PDF parte 2, págs. 55–93

[Índice de la semana](../semana-03.ipynb) | [Anterior](u25-busqueda-binaria.ipynb) | [Siguiente](../repaso.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Objetivos de la unidad

- Comprender los problemas de hashing y resolver problemas de búsqueda utilizando una tabla hash.
- Utilizar la estructura de datos de diccionario en el lenguaje Python.
- Implementar una tabla hash en Python y aplicarla a la resolución de problemas.
- Guardar pares clave-valor y realizar búsquedas con un diccionario; implementar una tabla hash en una clase de Python; buscar datos en tiempo constante.
- Utilizar un diccionario para construir un conversor de números romanos.

Palabras clave del PDF: tabla hash, hashing, función hash, diccionario, tiempo constante.

Conocimientos previos que pide el PDF: guardar y buscar datos con valores clave en un diccionario, definir variables y métodos en una clase, y expresar la complejidad temporal con notación Big-O.

**Referencia:** PDF parte 2, págs. 55–57

## Contexto

Las unidades anteriores buscaban un dato recorriéndolo (búsqueda secuencial) o dividiendo el rango en mitades (búsqueda binaria, que exige datos ordenados). La tabla hash propone otra idea: **calcular directamente** dónde está guardado el dato a partir de su clave, de modo que buscar cueste un tiempo casi constante sin importar cuántos datos haya.

El PDF lo ilustra con una estantería de 8 ranuras donde se colocan 5 libros, y lo aplica en un problema real: un conversor de números romanos a números arábigos. En la antigua Roma se usaban los números romanos; la misión es crear un programa que los convierta a números arábigos usando un diccionario de Python, que internamente es una tabla hash.

**Referencia:** PDF parte 2, págs. 59 y 69

## 1. Definición de tabla hash

Una **tabla hash** (o mapa hash) es un tipo de dato abstracto que almacena y extrae datos asignando claves a valores mediante *hashing*. El **hashing** consiste en designar la ubicación donde se guarda un dato (o donde se busca un dato guardado) a partir del valor del propio dato. La **función hash** determina esa ubicación a partir de la clave.

El PDF indica que la tabla hash debe proporcionar tres interfaces:

- `put(clave, valor)`: almacena el valor en el cubo (bucket) designado por el valor hash de la clave.
- `get(clave)`: extrae el valor almacenado en el cubo designado por el valor hash de la clave.
- `hash(clave)`: convierte el valor de la clave de los datos en un valor hash dentro de un rango específico.

El esquema del PDF muestra una tabla con 8 posiciones (`hashkey` de 0 a 7); el par `(clave, valor)` pasa por la función hash, que calcula `hashkey = hash(clave)` y decide en qué posición queda guardado.

**Referencia:** PDF parte 2, págs. 67–68

**Ampliación (no está en el PDF):** antes de ver la clase del PDF, esta es la idea mínima con una lista de 8 posiciones. La función `hash_simple` convierte una clave numérica en una posición (`hashkey`) entre 0 y 7.

In [2]:
# Ampliación: tabla hash mínima con una lista de 8 ranuras
TAMANO = 8
ranuras = [None] * TAMANO          # cada posición (hashkey) guarda un par (clave, valor)

def hash_simple(clave):
    """Convierte una clave numérica en una posición entre 0 y TAMANO - 1."""
    return clave % TAMANO

def put_simple(clave, valor):
    ranuras[hash_simple(clave)] = (clave, valor)

def get_simple(clave):
    par = ranuras[hash_simple(clave)]
    return par[1] if par is not None and par[0] == clave else None

put_simple(1584, "El Principito")
print("hashkey de 1584:", hash_simple(1584))
print("get(1584):", get_simple(1584))
print("get(999):", get_simple(999))
print(ranuras)

hashkey de 1584: 0
get(1584): El Principito
get(999): None
[(1584, 'El Principito'), None, None, None, None, None, None, None]


**Clave:** la tabla no recorre los datos: calcula la posición con la función hash y va directo a ella.

**Para el quiz:** las tres interfaces de una tabla hash son `put(clave, valor)`, `get(clave)` y `hash(clave)`.

**Referencia:** PDF parte 2, págs. 67–68

## 2. Ejemplo de tabla hash: la estantería de libros

El PDF imagina 5 libros que se colocan en una estantería con 8 ranuras: *El Principito*, *El viejo y el mar*, *La Sirenita*, *La Bella y la Bestia* y *La última hoja*. A cada libro se le asigna un valor clave. Aquí se usa **la suma de los códigos ASCII del título**. La función `ord` devuelve el código ASCII de una letra, y `map` la aplica a cada letra del título.

In [3]:
# Códigos ASCII de cada carácter del título (ejemplo del PDF)
print(list(map(ord, "The Last Leaf")))
print(sum(map(ord, "The Last Leaf")))

[84, 104, 101, 32, 76, 97, 115, 116, 32, 76, 101, 97, 102]
1133


**Nota sobre el PDF:** la diapositiva de la pág. 70 muestra la suma "= 1,113", pero la suma real de esos códigos es **1133** (la propia salida de la pág. 71 y la pág. 81 muestran 1133). El diagrama de la pág. 72 también rotula 1113. En este notebook se usa el valor correcto, 1133.

Ahora el cálculo para los cinco libros, como en el PDF (los títulos están en inglés en el código):

In [4]:
books = [
    "The Little Prince",
    "The Old Man and the Sea",
    "The Little Mermaid",
    "Beauty and the Beast",
    "The Last Leaf"
]
for book in books:
    key = sum(map(ord, book))   # valor clave del libro = suma de códigos ASCII
    print(key, book)

1584 The Little Prince
1929 The Old Man and the Sea
1678 The Little Mermaid
1837 Beauty and the Beast
1133 The Last Leaf


**Clave:** la suma de los códigos ASCII de todas las letras es el valor clave del libro. Es una función determinista: el mismo título siempre produce el mismo número.

**Referencia:** PDF parte 2, págs. 69–71

## 3. Hashing y funciones hash

Una **función hash** convierte una clave de longitud arbitraria en otra clave de longitud fija. La clave convertida se llama **clave hash** (`hashkey`). Como la estantería tiene 8 ranuras, el PDF define la función hash como `f(x) = x % 8`: el resto de dividir la clave entre 8 es el número de ranura (0 a 7).

In [5]:
# f(x) = x % 8 aplicada a las claves de los libros
for book in books:
    key = sum(map(ord, book))
    print(f"{book:<25} clave={key}  hashkey={key % 8}")

The Little Prince         clave=1584  hashkey=0
The Old Man and the Sea   clave=1929  hashkey=1
The Little Mermaid        clave=1678  hashkey=6
Beauty and the Beast      clave=1837  hashkey=5
The Last Leaf             clave=1133  hashkey=5


**Colisiones (Ampliación, no está en el PDF como concepto explícito):** observa que "Beauty and the Beast" (1837) y "The Last Leaf" (1133) dan ambos la ranura 5. Cuando dos claves distintas producen la misma clave hash hay una **colisión**. El PDF la resuelve sin nombrarla: cada ranura es una **lista** (cubo) que puede guardar varios libros, y su salida de la pág. 80 muestra `['Beauty and the Beast', 'The Last Leaf']` en el cubo 5. Esta técnica se llama encadenamiento (chaining).

**Nota sobre el PDF:** el diagrama de la pág. 72 usa 1113 en lugar de 1133. Con 1113 también habría colisión (1113 % 8 = 1, igual que 1929); con el valor correcto la colisión ocurre en la ranura 5.

**Ampliación (no está en el PDF):** la función `hash()` incorporada de Python no sirve para estos ejemplos con salida, porque para las cadenas cambia entre ejecuciones del intérprete (aleatorización controlada por `PYTHONHASHSEED`, una medida de seguridad). La suma de códigos ASCII que usa el PDF es determinista y por eso se usa aquí como función hash propia. Se puede comprobar ejecutando Python dos veces con semillas distintas.

In [6]:
import subprocess
import sys

def hash_en_proceso_nuevo(semilla):
    """Ejecuta hash('The Last Leaf') en un intérprete nuevo con una semilla dada."""
    resultado = subprocess.run(
        [sys.executable, "-c", "print(hash('The Last Leaf'))"],
        env={"PYTHONHASHSEED": str(semilla)},
        capture_output=True, text=True,
    )
    return resultado.stdout.strip()

# Con semillas distintas el hash de la misma cadena es distinto
print("Semillas distintas dan valores distintos:", hash_en_proceso_nuevo(1) != hash_en_proceso_nuevo(2))
# Con la misma semilla es reproducible
print("Misma semilla da el mismo valor:", hash_en_proceso_nuevo(1) == hash_en_proceso_nuevo(1))

# En cambio, la función del PDF es determinista
print("Suma ASCII:", sum(map(ord, "The Last Leaf")), "-> siempre 1133")

Semillas distintas dan valores distintos: True
Misma semilla da el mismo valor: True
Suma ASCII: 1133 -> siempre 1133


**Clave:** una función hash transforma una clave en una posición fija del rango de la tabla; dos claves distintas pueden caer en la misma posición.

**Error común:** creer que cada clave tiene su propia ranura. Si hay más claves que ranuras (o simplemente mala suerte), habrá colisiones y la tabla debe saber manejarlas.

**Para el quiz:** `f(x) = x % 8` es la función hash del PDF; la clave convertida se llama clave hash (hashkey).

**Referencia:** PDF parte 2, págs. 72 y 80

## 4. Definir una tabla hash como tipo de clase

El PDF implementa la tabla hash como la clase `HashTable`:

- `__init__(self, size)`: recibe el tamaño de la tabla. La tabla se define con un diccionario de Python y cada cubo se inicializa como una lista vacía (líneas 3-7).
- `hash(self, key)`: devuelve `key % self.size`.
- `get(self, key)`: devuelve el cubo cuya clave hash es `self.hash(key)` (líneas 12-13).
- `put(self, key, value)`: busca el cubo en `self.table` y, como no se permite duplicación, añade el valor solo si el cubo todavía no lo tiene (líneas 15-18).

In [7]:
class HashTable:

    def __init__(self, size):
        self.size = size
        self.table = {}
        for i in range(size):
            self.table[i] = []

    def hash(self, key):
        return key % self.size

    def get(self, key):
        return self.table[self.hash(key)]

    def put(self, key, value):
        bucket = self.table[self.hash(key)]
        if value not in bucket:
            bucket.append(value)

**Nota sobre el PDF:** en la definición de la interfaz (pág. 67) `get(clave)` extrae "el valor", pero en la clase `get` devuelve el **cubo completo** (una lista), porque en un cubo puede haber varios valores colisionados. Para saber si un libro concreto está, hay que buscarlo dentro de la lista devuelta (ver el Ejercicio 4).

**Clave:** `self.table` es un diccionario cuyas claves son las posiciones 0 a `size - 1` y cuyos valores son listas (cubos). `put` y `get` usan `self.hash(key)` para llegar directo al cubo.

**Error común:** olvidar que `put` ignora valores repetidos en el mismo cubo (`if value not in bucket`): guardar dos veces el mismo libro no lo duplica.

**Referencia:** PDF parte 2, págs. 73–77

## 5. Ejemplo de utilización de la tabla hash

Primero se inicializa una tabla con 8 cubos y se muestran todos vacíos.

In [8]:
table = HashTable(8)
for key in table.table.keys():
    print(key, table.table[key])

0 []
1 []
2 []
3 []
4 []
5 []
6 []
7 []


Se añade "El Principito" (en el código, `"The Little Prince"`). Su clave es 1584 y 1584 % 8 = 0, así que se almacena en el cubo 0.

In [9]:
table = HashTable(8)
book = "The Little Prince"
key = sum(map(ord, book))
table.put(key, book)
for key in table.table.keys():
    print(key, table.table[key])

0 ['The Little Prince']
1 []
2 []
3 []
4 []
5 []
6 []
7 []


Se colocan los otros cuatro libros. Observa el cubo 5, donde conviven dos libros (colisión resuelta por el cubo-lista).

In [10]:
books = [
    "The Old Man and the Sea",
    "The Little Mermaid",
    "Beauty and the Beast",
    "The Last Leaf"
]
for book in books:
    key = sum(map(ord, book))
    table.put(key, book)
for key in table.table.keys():
    print(key, table.table[key])

0 ['The Little Prince']
1 ['The Old Man and the Sea']
2 []
3 []
4 []
5 ['Beauty and the Beast', 'The Last Leaf']
6 ['The Little Mermaid']
7 []


Para encontrar la ranura donde está "La última hoja" se usa `get()`:

In [11]:
title = "The Last Leaf"
key = sum(map(ord, title))
bucket = table.get(key)
print(key, bucket)

1133 ['Beauty and the Beast', 'The Last Leaf']


**Clave:** `get` entrega el cubo completo; el libro buscado está dentro de esa lista, junto con los que colisionaron.

**Referencia:** PDF parte 2, págs. 78–81

### Prueba sorpresa del PDF (Cuestionario 1 y 2)

El PDF propone dos ejercicios: calcular la clave y la clave hash de "Alice in WonderLand" con una tabla de 8 ranuras, y ver en qué hueco queda cada libro con una estantería de 10 huecos. Ejecutados, dan lo siguiente.

In [12]:
# Cuestionario 1 del PDF: clave y clave hash de "Alice in WonderLand" (tabla de 8)
table = HashTable(8)
book = "Alice in WonderLand"
key = sum(map(ord, book))
print(key, table.hash(key))

1763 3


In [13]:
# Cuestionario 2 del PDF: estantería de 10 huecos con seis libros
table = HashTable(10)
books = [
    "The Little Prince",
    "The Old Man and the Sea",
    "The Little Mermaid",
    "Beauty and the Beast",
    "The Last Leaf",
    "Alice in WonderLand"
]
for book in books:
    key = sum(map(ord, book))
    table.put(key, book)
for key in table.table.keys():
    print(key, table.table[key])

0 []
1 []
2 []
3 ['The Last Leaf', 'Alice in WonderLand']
4 ['The Little Prince']
5 []
6 []
7 ['Beauty and the Beast']
8 ['The Little Mermaid']
9 ['The Old Man and the Sea']


**Referencia:** PDF parte 2, págs. 88–89

## 6. Tabla hash con diccionario y tiempo constante

Un diccionario de Python (`dict`) puede considerarse una tabla hash (o mapa hash): el PDF lo usa para convertir letras romanas en números con su "código hash".

```python
table = {'I':1, 'V':5, 'X':10, 'L':50, 'C':100, 'D':500, 'M':1000}
```

Aquí la clave es la letra y el valor es su número; Python calcula internamente la posición, así que `table['M']` es una consulta directa.

In [14]:
table = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
print(table['M'])
print(table['X'])
print('Z' in table)

1000
10
False


**Ampliación (no está en el PDF):** el PDF incluye la palabra clave "tiempo constante". Con una buena función hash, `put` y `get` en promedio cuestan O(1): no dependen de cuántos datos haya. En el peor caso (todas las claves caen en el mismo cubo) la búsqueda dentro del cubo es lineal, O(n). Se puede comprobar con una tabla de tamaño 1: todo cae en el cubo 0.

In [15]:
# Ampliación: peor caso, una tabla de un solo cubo
peor = HashTable(1)
for libro in ["A", "B", "C", "D"]:
    peor.put(sum(map(ord, libro)), libro)
print(peor.table)     # los 4 valores quedan en el mismo cubo

# Mejor caso: tabla amplia, cada valor casi siempre en su propio cubo
amplia = HashTable(1000)
for libro in ["A", "B", "C", "D"]:
    amplia.put(sum(map(ord, libro)), libro)
print([k for k, v in amplia.table.items() if v])   # cubos ocupados

{0: ['A', 'B', 'C', 'D']}
[65, 66, 67, 68]


**Clave:** un diccionario es una tabla hash lista para usar; su ventaja es la búsqueda en tiempo (promedio) constante.

**Error común:** pensar que "tiempo constante" garantiza una sola operación siempre. Depende de que la función hash reparta bien las claves.

**Para el quiz:** palabras clave de la unidad: tabla hash, hashing, función hash, diccionario, tiempo constante.

**Referencia:** PDF parte 2, págs. 56–57 y 83

## Proyecto de la unidad: conversor de números romanos a arábigos

**Problema.** Crear un conversor que reciba un número romano y lo imprima como número arábigo. Los números romanos usan 7 símbolos distintos: I=1, V=5, X=10, L=50, C=100, D=500, M=1000. Normalmente se suman los símbolos en orden (III es 3, XVI es 10 + 5 + 1 = 16). Pero si el símbolo siguiente es mayor, se **resta** el valor del símbolo actual (IX es 10 - 1 = 9, IV es 5 - 1 = 4).

Ejemplos del PDF:

- MDCLXVI: 1000 + 500 + 100 + 50 + 10 + 5 + 1 = 1666.
- MCDXLIV: 1000 - 100 + 500 - 10 + 50 - 1 + 5 = 1444.
- MCMXCIX: 1000 - 100 + 1000 - 10 + 100 - 1 + 10 = 1999.

**Plan de programación (pasos del PDF):**

1. Crear una tabla hash (diccionario) con el valor de cada símbolo.
2. Leer el número romano con `input` (en el notebook, se encapsula en una función).
3. Iniciar `result` en 0 y recorrer todas las letras menos la última: si el valor de la letra actual es menor que el de la siguiente, restarlo; si no, sumarlo.
4. Al terminar el bucle, devolver `result` más el valor de la última letra (`str[-1]`).

**Referencia:** PDF parte 2, págs. 59–65 y 83–86

### Código del PDF

El PDF define la tabla y la función `roman_to_int`. La función se ejecuta con los tres ejemplos de las págs. 61–63 (en el PDF se pide la cadena con `input`; aquí se pasa directamente).

In [16]:
table = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}

def roman_to_int(str):
    result = 0
    for i in range(len(str) - 1):
        if table[str[i]] < table[str[i + 1]]:
            result -= table[str[i]]      # el siguiente es mayor: se resta
        else:
            result += table[str[i]]      # si no, se suma
    return result + table[str[-1]]       # la última letra siempre se suma

for roman in ["MDCLXVI", "MCDXLIV", "MCMXCIX"]:
    number = roman_to_int(roman)
    print("Input a Roman number:", roman)
    print(number)

Input a Roman number: MDCLXVI
1666
Input a Roman number: MCDXLIV
1444
Input a Roman number: MCMXCIX
1999


**Nota sobre el PDF:** el parámetro se llama `str`, lo que oculta (sombrea) la función incorporada `str` dentro de la función; funciona, pero es mala práctica. Además, la pág. 63 escribe "MCMSCIX" en el texto; el número del esquema y de la salida es MCMXCIX = 1999. La función del PDF falla con una cadena vacía (`str[-1]` lanza `IndexError`) y con letras que no estén en la tabla (`KeyError`).

**Ampliación (no está en el PDF):** versión más robusta con validación, y una función interactiva que termina cuando el usuario cancela (entrada vacía o `None`).

In [17]:
def romano_a_entero(texto):
    """Convierte un número romano (cadena) a entero. Lanza ValueError si la entrada no es válida."""
    texto = (texto or "").strip().upper()
    if not texto:
        raise ValueError("La entrada está vacía")
    for letra in texto:
        if letra not in table:
            raise ValueError(f"Símbolo no válido: {letra!r}")
    resultado = 0
    for i in range(len(texto) - 1):
        if table[texto[i]] < table[texto[i + 1]]:
            resultado -= table[texto[i]]
        else:
            resultado += table[texto[i]]
    return resultado + table[texto[-1]]


def conversor_interactivo(leer=input):
    """Pide números romanos hasta que el usuario cancele (entrada vacía o None)."""
    while True:
        try:
            texto = leer("Ingrese un número Romano (vacío para terminar): ")
        except (EOFError, KeyboardInterrupt):
            texto = None
        if texto is None or not texto.strip():
            print("Fin del conversor.")
            break
        try:
            print(romano_a_entero(texto))
        except ValueError as error:
            print(f"Error: {error}")


# Versión no interactiva: se simulan las entradas con una lista de comandos de prueba
entradas = iter(["MDCLXVI", "mcdxliv", "MCMXCIX", "ABC", "", "XIV"])   # la entrada vacía termina
conversor_interactivo(lambda mensaje: next(entradas, None))

# Para usarlo de forma interactiva, quita el comentario de la siguiente línea
# conversor_interactivo()

1666
1444
1999
Error: Símbolo no válido: 'A'
Fin del conversor.


Se comprueban los ejemplos de la pág. 93 (romanos que luego se piden en el sentido inverso):

In [18]:
ejemplos = {
    369: "CCCLXIX", 80: "LXXX", 29: "XXIX", 155: "CLV", 14: "XIV",
    492: "CDXCII", 348: "CCCXLVIII", 301: "CCCI", 469: "CDLXIX", 499: "CDXCIX",
}
for numero, romano in ejemplos.items():
    verificar(f"{romano} -> {numero}", roman_to_int(romano), numero)

Correcto: CCCLXIX -> 369
Correcto: LXXX -> 80
Correcto: XXIX -> 29
Correcto: CLV -> 155
Correcto: XIV -> 14
Correcto: CDXCII -> 492
Correcto: CCCXLVIII -> 348
Correcto: CCCI -> 301
Correcto: CDLXIX -> 469
Correcto: CDXCIX -> 499


**Resultado esperado:** MDCLXVI da 1666, MCDXLIV da 1444 y MCMXCIX da 1999, y los diez romanos de la pág. 93 coinciden con sus números.

**Referencia:** PDF parte 2, págs. 59–65 y 83–86 y 93

### Programación en pareja: de arábigos a romanos

El PDF cierra con una práctica de programación en pareja: crear un conversor que convierta números arábigos en romanos con esta tabla hash, y probar con los ejemplos 369, 80, 29, 155, 14, 492, 348, 301, 469 y 499 (romanos: CCCLXIX, LXXX, XXIX, CLV, XIV, CDXCII, CCCXLVIII, CCCI, CDLXIX, CDXCIX). La salida esperada para 1999 es MCMXCIX.

**Nota sobre el PDF:** la tabla de la pág. 93 contiene `4:'XI'`, que es un error: el 4 se escribe `'IV'`. Aquí se corrige. Además, el orden de la tabla importa: debe ir de mayor a menor valor.

Las págs. 91–92 describen buenas prácticas de programación en pareja: turnarse (quien maneja el mouse), emparejar habilidades similares pero no necesariamente iguales, entender que no es "divide y vencerás" sino un trabajo colaborativo, ofrecer incentivos y evitar trampas evaluando los resultados individuales.

Plan: recorrer la tabla de mayor a menor; mientras el número restante sea al menos el valor, añadir el símbolo y restar el valor.

In [19]:
tabla_inversa = {1000: 'M', 900: 'CM', 500: 'D', 400: 'CD',
                 100: 'C', 90: 'XC', 50: 'L', 40: 'XL',
                 10: 'X', 9: 'IX', 5: 'V', 4: 'IV', 1: 'I'}   # corregido: 4 -> 'IV'

def int_to_roman(num):
    resultado = ""
    for valor, simbolo in tabla_inversa.items():
        while num >= valor:
            resultado += simbolo
            num -= valor
    return resultado


def conversor_inverso_interactivo(leer=input):
    """Pide números arábigos hasta que el usuario cancele (entrada vacía o None)."""
    while True:
        try:
            texto = leer("Ingrese un número (vacío para terminar): ")
        except (EOFError, KeyboardInterrupt):
            texto = None
        if texto is None or not texto.strip():
            print("Fin del conversor.")
            break
        try:
            num = int(texto)
        except ValueError:
            print("Error: debe ingresar un número entero")
            continue
        if not 1 <= num <= 3999:
            print("Error: el número debe estar entre 1 y 3999")
            continue
        print(int_to_roman(num))


print(int_to_roman(1999))
for numero, romano in ejemplos.items():
    verificar(f"{numero} -> {romano}", int_to_roman(numero), romano)

# Versión no interactiva con comandos de prueba
entradas = iter(["1999", "abc", "0", "4", ""])
conversor_inverso_interactivo(lambda mensaje: next(entradas, None))

# Para usarlo de forma interactiva, quita el comentario de la siguiente línea
# conversor_inverso_interactivo()

MCMXCIX
Correcto: 369 -> CCCLXIX
Correcto: 80 -> LXXX
Correcto: 29 -> XXIX
Correcto: 155 -> CLV
Correcto: 14 -> XIV
Correcto: 492 -> CDXCII
Correcto: 348 -> CCCXLVIII
Correcto: 301 -> CCCI
Correcto: 469 -> CDLXIX
Correcto: 499 -> CDXCIX
MCMXCIX
Error: debe ingresar un número entero
Error: el número debe estar entre 1 y 3999
IV
Fin del conversor.


**Referencia:** PDF parte 2, págs. 91–93

## Práctica

Los ejercicios usan la clase `HashTable`, el diccionario `table` de romanos y las funciones definidas arriba. Completa cada función; la celda siguiente la comprueba. Las celdas sin resolver muestran "Incorrecto" y eso es normal.

### Ejercicio 1 — Clave de un título

Escribe `clave_ascii(titulo)` que devuelva la suma de los códigos ASCII de todos los caracteres del título.

In [20]:
def clave_ascii(titulo):
    # Escribe tu solución aquí
    pass

In [21]:
verificar("clave de 'The Last Leaf'", clave_ascii("The Last Leaf"), 1133)
verificar("clave de 'A'", clave_ascii("A"), 65)
verificar("clave de cadena vacía", clave_ascii(""), 0)

Incorrecto: clave de 'The Last Leaf'. Se esperaba 1133 y se obtuvo None
Incorrecto: clave de 'A'. Se esperaba 65 y se obtuvo None
Incorrecto: clave de cadena vacía. Se esperaba 0 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def clave_ascii(titulo):
    return sum(map(ord, titulo))
```

`map(ord, titulo)` produce el código de cada carácter y `sum` los suma. Una cadena vacía suma 0.
</details>

### Ejercicio 2 — Ranura de un título

Escribe `ranura(titulo, tamano)` que devuelva la clave hash (posición) del título en una tabla de `tamano` ranuras usando `suma de ASCII % tamano`.

In [22]:
def ranura(titulo, tamano):
    # Escribe tu solución aquí
    pass

In [23]:
verificar("'The Little Prince' en 8", ranura("The Little Prince", 8), 0)
verificar("'The Last Leaf' en 8", ranura("The Last Leaf", 8), 5)
verificar("'Alice in WonderLand' en 10", ranura("Alice in WonderLand", 10), 3)

Incorrecto: 'The Little Prince' en 8. Se esperaba 0 y se obtuvo None
Incorrecto: 'The Last Leaf' en 8. Se esperaba 5 y se obtuvo None
Incorrecto: 'Alice in WonderLand' en 10. Se esperaba 3 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def ranura(titulo, tamano):
    return sum(map(ord, titulo)) % tamano
```

El resto de la división entre el tamaño siempre cae entre 0 y `tamano - 1`.
</details>

### Ejercicio 3 — Contar colisiones

Escribe `ranuras_con_colision(titulos, tamano)` que devuelva cuántas ranuras terminan con más de un título. Usa `HashTable`.

In [24]:
def ranuras_con_colision(titulos, tamano):
    # Escribe tu solución aquí
    pass

In [25]:
cinco = ["The Little Prince", "The Old Man and the Sea", "The Little Mermaid",
         "Beauty and the Beast", "The Last Leaf"]
verificar("cinco libros en 8 ranuras", ranuras_con_colision(cinco, 8), 1)
verificar("cinco libros en 1 ranura", ranuras_con_colision(cinco, 1), 1)
verificar("sin libros", ranuras_con_colision([], 8), 0)

Incorrecto: cinco libros en 8 ranuras. Se esperaba 1 y se obtuvo None
Incorrecto: cinco libros en 1 ranura. Se esperaba 1 y se obtuvo None
Incorrecto: sin libros. Se esperaba 0 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def ranuras_con_colision(titulos, tamano):
    tabla = HashTable(tamano)
    for titulo in titulos:
        tabla.put(sum(map(ord, titulo)), titulo)
    return sum(1 for cubo in tabla.table.values() if len(cubo) > 1)
```

Se insertan todos los títulos y se cuentan los cubos con más de un elemento.
</details>

### Ejercicio 4 — Buscar un libro

Escribe `contiene(tabla, titulo)` que indique si el título está guardado en una `HashTable`. Recuerda que `get` devuelve el cubo completo.

In [26]:
def contiene(tabla, titulo):
    # Escribe tu solución aquí
    pass

In [27]:
estante = HashTable(8)
for t in ["The Little Prince", "Beauty and the Beast", "The Last Leaf"]:
    estante.put(sum(map(ord, t)), t)
verificar("libro presente en cubo compartido", contiene(estante, "The Last Leaf"), True)
verificar("libro presente", contiene(estante, "The Little Prince"), True)
verificar("libro ausente", contiene(estante, "Alice in WonderLand"), False)

Incorrecto: libro presente en cubo compartido. Se esperaba True y se obtuvo None
Incorrecto: libro presente. Se esperaba True y se obtuvo None
Incorrecto: libro ausente. Se esperaba False y se obtuvo None


<details><summary>Ver solución</summary>

```python
def contiene(tabla, titulo):
    cubo = tabla.get(sum(map(ord, titulo)))
    return titulo in cubo
```

Se calcula la clave, se obtiene el cubo con `get` y se busca el título dentro de la lista (el cubo es pequeño).
</details>

### Ejercicio 5 — De romano a arábigo

Escribe `romano_a_arabigo(texto)` que convierta un número romano válido a entero usando el diccionario `table`.

In [28]:
def romano_a_arabigo(texto):
    # Escribe tu solución aquí
    pass

In [29]:
verificar("MDCLXVI", romano_a_arabigo("MDCLXVI"), 1666)
verificar("MCMXCIX", romano_a_arabigo("MCMXCIX"), 1999)
verificar("IV", romano_a_arabigo("IV"), 4)

Incorrecto: MDCLXVI. Se esperaba 1666 y se obtuvo None
Incorrecto: MCMXCIX. Se esperaba 1999 y se obtuvo None
Incorrecto: IV. Se esperaba 4 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def romano_a_arabigo(texto):
    total = 0
    for i in range(len(texto) - 1):
        if table[texto[i]] < table[texto[i + 1]]:
            total -= table[texto[i]]
        else:
            total += table[texto[i]]
    return total + table[texto[-1]]
```

Se resta la letra si la siguiente vale más; si no, se suma. La última letra siempre se suma.
</details>

### Ejercicio 6 — De arábigo a romano

Escribe `arabigo_a_romano(n)` que convierta un entero entre 1 y 3999 a número romano usando una tabla de mayor a menor con los casos de resta (CM, CD, XC, XL, IX, IV).

In [30]:
def arabigo_a_romano(n):
    # Escribe tu solución aquí
    pass

In [31]:
verificar("1999", arabigo_a_romano(1999), "MCMXCIX")
verificar("14", arabigo_a_romano(14), "XIV")
verificar("499", arabigo_a_romano(499), "CDXCIX")

Incorrecto: 1999. Se esperaba 'MCMXCIX' y se obtuvo None
Incorrecto: 14. Se esperaba 'XIV' y se obtuvo None
Incorrecto: 499. Se esperaba 'CDXCIX' y se obtuvo None


<details><summary>Ver solución</summary>

```python
def arabigo_a_romano(n):
    tabla = {1000: 'M', 900: 'CM', 500: 'D', 400: 'CD', 100: 'C', 90: 'XC',
             50: 'L', 40: 'XL', 10: 'X', 9: 'IX', 5: 'V', 4: 'IV', 1: 'I'}
    resultado = ""
    for valor, simbolo in tabla.items():
        while n >= valor:
            resultado += simbolo
            n -= valor
    return resultado
```

El diccionario conserva el orden de inserción, de mayor a menor. Por cada valor se añade el símbolo mientras quepa.
</details>

### Ejercicio 7 — Validar un número romano

Escribe `es_romano_valido(texto)` que devuelva `True` solo si el texto está escrito en forma canónica: usa solo símbolos válidos y, al convertirlo a entero y de vuelta a romano, se obtiene el mismo texto. Por ejemplo, `IIII` y `IC` no son canónicos. Puedes usar `romano_a_entero` y `int_to_roman` del proyecto.

In [32]:
def es_romano_valido(texto):
    # Escribe tu solución aquí
    pass

In [33]:
verificar("MCMXCIX", es_romano_valido("MCMXCIX"), True)
verificar("IIII", es_romano_valido("IIII"), False)
verificar("ABC", es_romano_valido("ABC"), False)
verificar("IC", es_romano_valido("IC"), False)

Incorrecto: MCMXCIX. Se esperaba True y se obtuvo None
Incorrecto: IIII. Se esperaba False y se obtuvo None
Incorrecto: ABC. Se esperaba False y se obtuvo None
Incorrecto: IC. Se esperaba False y se obtuvo None


<details><summary>Ver solución</summary>

```python
def es_romano_valido(texto):
    if not texto or any(letra not in table for letra in texto):
        return False
    return int_to_roman(romano_a_entero(texto)) == texto
```

Se descartan vacíos y símbolos desconocidos. Después se convierte a entero y de vuelta: solo las formas canónicas regresan idénticas.
</details>

## Apuntes para estudiar

**Clave**

- Una tabla hash guarda y busca datos asignando claves a valores mediante hashing; sus interfaces son `put(clave, valor)`, `get(clave)` y `hash(clave)`.
- La función hash convierte una clave en una clave hash (posición). En el ejemplo de la estantería: `f(x) = x % 8` sobre la suma de códigos ASCII del título.
- La clase `HashTable` del PDF usa un diccionario cuyas posiciones 0 a `size - 1` guardan listas (cubos); `put` agrega si el valor no está y `get` devuelve el cubo.
- Dos claves distintas pueden caer en la misma ranura (colisión): el cubo-lista las guarda juntas.
- Un diccionario de Python es una tabla hash lista para usar con búsqueda en tiempo (promedio) constante.
- Conversor romano a arábigo: se resta el valor de la letra si la siguiente vale más; si no, se suma; la última siempre se suma.

**Errores comunes**

- Creer que `get` devuelve un valor: en la clase del PDF devuelve el cubo completo (lista).
- Olvidar que valores duplicados se ignoran en `put`.
- Confiar en `hash()` de cadenas para salidas reproducibles: cambia entre ejecuciones por `PYTHONHASHSEED`.
- Usar `str` como nombre de parámetro (oculta la función incorporada) o no validar la cadena vacía en el conversor.
- Escribir la tabla inversa fuera de orden o con el error `4:'XI'` del PDF (debe ser `'IV'`).

**Para el quiz**

- Palabras clave: tabla hash, hashing, función hash, diccionario, tiempo constante.
- Valores del ejemplo: "The Little Prince" = 1584 (cubo 0), "The Old Man and the Sea" = 1929 (cubo 1), "The Little Mermaid" = 1678 (cubo 6), "Beauty and the Beast" = 1837 y "The Last Leaf" = 1133 (ambos cubo 5).
- MDCLXVI = 1666, MCDXLIV = 1444, MCMXCIX = 1999.
- Con 10 huecos, "Alice in WonderLand" (clave 1763) va al hueco 3.

## Autoevaluación

**1. ¿Qué tres interfaces debe proporcionar una tabla hash según el PDF?**

<details><summary>Ver respuesta</summary>

`put(clave, valor)` para almacenar, `get(clave)` para extraer y `hash(clave)` para convertir la clave en un valor hash dentro de un rango específico.
</details>

**2. ¿Qué ocurre cuando dos claves distintas producen la misma clave hash y cómo lo resuelve la clase del PDF?**

<details><summary>Ver respuesta</summary>

Es una colisión. La clase la resuelve porque cada ranura es un cubo (lista) que puede contener varios valores; por ejemplo, el cubo 5 guarda "Beauty and the Beast" y "The Last Leaf".
</details>

**3. ¿Por qué `hash("texto")` de Python no es adecuado para ejemplos con salida reproducible, y qué se usa en su lugar en el PDF?**

<details><summary>Ver respuesta</summary>

Para cadenas, `hash()` cambia entre ejecuciones por la aleatorización de `PYTHONHASHSEED`. El PDF usa una función determinista: la suma de los códigos ASCII (`sum(map(ord, titulo))`) y luego `% tamaño`.
</details>

**4. ¿Por qué un diccionario de Python puede considerarse una tabla hash y qué ventaja ofrece?**

<details><summary>Ver respuesta</summary>

Porque asocia claves con valores calculando internamente la posición de cada clave. Ofrece búsqueda en tiempo (promedio) constante, sin recorrer los elementos.
</details>

**5. En `roman_to_int`, ¿por qué se recorre hasta `len(str) - 1` y luego se suma `table[str[-1]]`?**

<details><summary>Ver respuesta</summary>

Porque en cada paso se compara la letra actual con la siguiente, y la última letra no tiene siguiente. Su valor siempre se suma al final, ya que nada posterior puede exigir restarla.
</details>

[Índice de la semana](../semana-03.ipynb) | [Anterior](u25-busqueda-binaria.ipynb) | [Siguiente](../repaso.ipynb)